# nuScenes devkit tutorial — applied to `tcar_nuscenes`

This mirrors the [official nuScenes devkit tutorial](https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb), keeping only the cells that work on **our label-prep dataset** (no annotations yet, no radar, custom map).

Skipped sections (will be enabled once annotations are populated):
- §4 sample_annotation, §5 instance, §7 attribute (instance walk), §8 visibility — all need annotations
- RADAR examples — we only have 6 standard cams + CAM_TRAFFIC + LIDAR_TOP
- `render_egoposes_on_map(log_location='singapore-onenorth')` — our log location is `korea-test` and map is a placeholder

## Initialization

In [ ]:
%matplotlib inline
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-trainval', dataroot='/data/tcar_nuscenes', verbose=True)

## 1. scene

In [ ]:
nusc.list_scenes()

In [ ]:
my_scene = nusc.scene[0]
my_scene

## 2. sample

In [ ]:
first_sample_token = my_scene['first_sample_token']

In [ ]:
my_sample = nusc.get('sample', first_sample_token)
my_sample

In [ ]:
nusc.list_sample(my_sample['token'])

## 3. sample_data

In [ ]:
my_sample['data']

In [ ]:
sensor = 'CAM_FRONT'
cam_front_data = nusc.get('sample_data', my_sample['data'][sensor])
cam_front_data

In [ ]:
nusc.render_sample_data(cam_front_data['token'])

## 9. sensor

In [ ]:
nusc.sensor

In [ ]:
nusc.sample_data[10]

## 10. calibrated_sensor

In [ ]:
nusc.calibrated_sensor[0]

Note: translation/rotation are with respect to the ego vehicle body frame.

## 11. ego_pose

In [ ]:
nusc.ego_pose[0]

Number of `ego_pose` records equals number of `sample_data` records (1-to-1).

## 12. log

In [ ]:
print("Number of `logs` in our loaded database: {}".format(len(nusc.log)))
nusc.log[0]

## 13. map

In [ ]:
print("There are {} maps masks in the loaded dataset".format(len(nusc.map)))
nusc.map[0]

## nuScenes Basics

In [ ]:
nusc.category[0]

In [ ]:
cat_token = nusc.category[0]['token']
cat_token

In [ ]:
nusc.get('category', cat_token)

### Shortcuts

The `sample_data` table has `channel` and `sensor_modality` shortcuts.

In [ ]:
# Shortcut
channel = nusc.sample_data[0]['channel']

# No shortcut
sd_rec = nusc.sample_data[0]
cs_record = nusc.get('calibrated_sensor', sd_rec['calibrated_sensor_token'])
sensor_record = nusc.get('sensor', cs_record['sensor_token'])

print(channel == sensor_record['channel'])

## Data Visualizations

### List methods

In [ ]:
nusc.list_categories()

In [ ]:
nusc.list_attributes()

In [ ]:
nusc.list_scenes()

### Render — lidar point cloud projected onto a camera image

In [ ]:
my_sample = nusc.sample[10]
nusc.render_pointcloud_in_image(my_sample['token'], pointsensor_channel='LIDAR_TOP')

Color by intensity instead of depth:

In [ ]:
nusc.render_pointcloud_in_image(my_sample['token'], pointsensor_channel='LIDAR_TOP', render_intensity=True)

### Single-sensor render

In [ ]:
my_sample = nusc.sample[20]
nusc.render_sample_data(my_sample['data']['CAM_FRONT'])

### 6-camera grid

Render the 6 standard NuScenes cameras for one sample in a 2×3 grid.
`render_sample_data(token, ax=...)` lets you place each render onto an existing matplotlib axes.

In [ ]:
import matplotlib.pyplot as plt

cams = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
        'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']

fig, axes = plt.subplots(2, 3, figsize=(20, 8))
for ax, cam in zip(axes.flat, cams):
    nusc.render_sample_data(my_sample['data'][cam], ax=ax)
    ax.set_title(cam)
    ax.axis('off')
plt.tight_layout()
plt.show()

### CAM_TRAFFIC (7th channel)

Our 7th channel for traffic light/sign. Calibration is currently a placeholder (identity transform), but the JPEG itself is properly synced to the keyframe.

In [ ]:
if 'CAM_TRAFFIC' in my_sample['data']:
    nusc.render_sample_data(my_sample['data']['CAM_TRAFFIC'])
else:
    print('CAM_TRAFFIC not present in this dataset.')

### Multi-sweep lidar BEV (denser cloud)

`underlay_map=False` because our map is a placeholder (devkit's map crop expects HD-map-sized raster keyed to ego coords).

In [ ]:
nusc.render_sample_data(my_sample['data']['LIDAR_TOP'], nsweeps=5, underlay_map=False)

### Full-sample render — skipped

`nusc.render_sample(token)` always uses `underlay_map=True` for the BEV panel; with our placeholder map, the BEV crop fails. Use `render_pointcloud_in_image` (above) and per-channel `render_sample_data` instead.

### Scene render

`nusc.render_scene` always opens an OpenCV native window via `cv2.namedWindow` (even when writing to file), which crashes a kernel without an X display. With a real GUI Jupyter session it works, but `nbconvert --execute` (and many remote/SSH setups) will kill the kernel.

For a headless-safe scene playback, see [`dataset_validation.ipynb`](dataset_validation.ipynb) §7 — we use `matplotlib.animation.FuncAnimation` + `to_jshtml()` to embed the scene as an inline HTML5 player.